# 20iv26_142407 — CNMF Time-Split, Full Resolution

**Run name:** `20iv26_142407_time_split_full`  
**Dataset:** `2026-04-20_142407` — `stack_0-A1-rGECO_channel_0_obj_bottom`  
**Indicator:** rGECO (red calcium indicator)  
**Mode:** `time-split` — Bayesian tuning on frames 0–25, held-out test on frames 26–52  
**Resolution:** `full` (2048 × 2048, no spatial downsampling)  
**Preprocessing:** stripe removal ON, brain mask ON  
**Quality filters:** circularity ≥ 0.5, max-area-factor 4.0, min-SNR 1.5, centroid in-mask

## Run Command

```bash
python p4_universal.py \\
    --mode time-split \\
    --data-dir /home/abl-dell/Downloads/caiman_dataset/2026-04-20_142407-20260610T194439Z-3-001/2026-04-20_142407/raw/stack_0-A1-rGECO_channel_0_obj_bottom \\
    --run-name 20iv26_142407_time_split_full \\
    --resolution full \\
    --n-workers 64 | python monitor.py --filename  142407_logs.txt
```

## Run at a Glance

| Property | Value |
|---|---|
| Input format | `single-movie` (1 file) |
| Raw shape | (53, 2048, 2048) |
| Final shape | (53, 2048, 2048) — no downsampling applied |
| Brain mask coverage | 22.1% of frame |
| Time split | tune: frames 0–25 (26 frames) · test: frames 26–52 (27 frames) |
| Bayesian trials | 10 (5 random initial) |
| Best tune composite | +0.0378 (Trial 4) |
| Test-half result | 32 raw → **2 kept** · runtime 485 s (~8 min) |
| Full-movie result | 104 raw → **5 kept** · runtime 2214 s (~37 min) |

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

RUN_DIR = Path("results/20iv26_142407_time_split_full")

with open(RUN_DIR / "summary.json") as f:
    summary = json.load(f)

---
## 1. Dataset & Preprocessing

In [ ]:
fmt = summary["format_info"]
print(f"Format         : {fmt['format']}")
print(f"Files          : {fmt['n_files']}")
print(f"Raw shape      : {fmt['sample_shape']}")
print(f"Final shape    : {fmt['final_shape']}")
print(f"Stripe removed : {fmt['stripe_removed']}")
print(f"Brain mask     : {fmt['brain_mask_used']}  "
      f"({fmt['mask_coverage_frac']*100:.1f}% of frame)")

### 1.1 Stripe Removal

Column-median subtraction (over all frames and rows) removes the vertical striping artifact introduced by the light-sheet illumination. The removed pattern (right panel) shows brightness concentrated in columns ~900–1300, corresponding to the brightest illumination band of the sheet.

In [ ]:
display(Image(str(RUN_DIR / "preprocess_movie.png"), width=1000))

### 1.2 Brain Mask

Otsu thresholding on the temporal mean image, followed by morphological opening/closing (disk radius proportional to image size) and small-object removal.

Coverage: **22.1%** of the 2048×2048 frame. The mask captures both cerebellar hemispheres with a narrow midline gap, plus two small satellite blobs at the tissue edges. The mask zeroes out the dark periphery before CNMF initialisation and rejects any component whose centroid falls outside the brain region during quality filtering.

In [ ]:
display(Image(str(RUN_DIR / "brain_mask_movie.png"), width=700))

---
## 2. Bayesian Hyperparameter Tuning

10 Gaussian-process trials (5 random + 5 GP-guided) over the **full-resolution** search space:

| Parameter | Range | Description |
|---|---|---|
| `gSig` | Integer [4, 16] | Neuron half-width (px) |
| `gSig_filt` | Integer [4, 16] | High-pass pre-filter half-width |
| `min_corr` | Real [0.50, 0.85] | Minimum local correlation to seed a component |
| `min_pnr` | Integer [5, 12] | Minimum peak-to-noise ratio to seed |
| `rf` | {100, 160, 240, 320} | Patch half-size (px) |
| `p` | {1, 2} | AR model order |

The composite score (maximised) is:
```
composite = 1.0*(1 - recon_error)
           + 0.5*spatial_compactness
           - 0.3*log(1 + trace_sparsity)
           + 1.0*stability
           + 0.001*log(1 + n_filtered)
```
All metrics are computed **after** quality filtering — the optimizer is rewarded for real neurons, not noise blobs.

In [ ]:
df = pd.read_csv(RUN_DIR / "tune_time_split_log.csv")
df.index = df.index + 1
df.index.name = "trial"

cols = ["gSig", "gSig_filt", "min_corr", "min_pnr", "rf", "p",
        "n_neurons_pre", "n_neurons", "composite_score", "runtime_s"]
df[cols].style \
    .format({"min_corr": "{:.3f}", "composite_score": "{:+.4f}",
             "runtime_s": "{:.0f}s"}) \
    .highlight_max(subset=["composite_score"], color="#b6d7a8") \
    .highlight_min(subset=["composite_score"], color="#ea9999")

**Trial 4** is the global winner (`gSig=4`, `gSig_filt=10`, `min_corr=0.640`, `min_pnr=5`, `rf=320`, `p=1`) with composite **+0.0378**.  

Trials 5 and 6 returned 0 components (scored −∞). The GP interpreted these as extreme negative regions and steered trials 7–10 toward the Trial 4 neighbourhood. All four converge to `rf=320`, `p=1` with `gSig` 10–11, but none improved on Trial 4.

### 2.1 Quality Filter Survival per Trial

In [ ]:
display(Image(str(RUN_DIR / "quality_filters_time_split.png"), width=950))

In [ ]:
df_counts = pd.read_csv(RUN_DIR / "quality_filters_time_split_log.csv")
int_cols = ["input", "circularity_rejected", "max_area_rejected",
            "in_mask_rejected", "final"]
df_counts = df_counts[["trial"] + int_cols].fillna(0)
df_counts[int_cols] = df_counts[int_cols].astype(int)
df_counts.set_index("trial")

**Max-area** is the dominant rejection reason in every trial. At `gSig=4` the area ceiling is `4 × π × 4² ≈ 201 px²` — very tight at 2048px. Even trials with larger `gSig` (10–15) lose the majority of components to oversized footprints, indicating CNMF is initialising neuropil patches as single-neuron candidates.

### 2.2 Convergence

In [ ]:
display(Image(str(RUN_DIR / "convergence_time_split.png"), width=950))

The optimizer converges after trial 4 with no further improvement across 6 remaining calls. 10 trials is insufficient for a 6-dimensional space — the search likely hit a local optimum early. Increasing `--n-calls 25 --n-initial 10` is recommended for future full-resolution runs.

### 2.3 Parameter vs Composite Score

In [ ]:
display(Image(str(RUN_DIR / "param_vs_score_time_split.png"), width=1050))

Directional signals from the scatter plots (n=10):

| Parameter | Winning value | Interpretation |
|---|---|---|
| `gSig` | **4** (smallest tested) | Neuron footprints are small relative to the 2048px FOV |
| `gSig_filt` | **10** (largest tested) | Strong high-pass filter needed to suppress broad background at full resolution |
| `min_corr` | ~**0.64** | Mid-range; >0.80 under-seeds, <0.55 seeds too many noise pixels |
| `min_pnr` | **5** (lower bound) | Modest per-pixel SNR; stricter PNR would miss most real seeds |
| `rf` | **320** | Larger patches better capture spatial structure of a 2048px frame |
| `p` | **1** | First-order AR sufficient; `p=2` added complexity without improvement |

---
## 3. Best Parameters

In [ ]:
bp = summary["best_params"]
pd.DataFrame(
    {"parameter": list(bp.keys()), "value": list(bp.values())}
).set_index("parameter")

---
## 4. Test Results

Best parameters are applied to two targets:
1. **Test half** (frames 26–52) — temporal generalization check
2. **Full movie** (frames 0–52) — final neuron map for the recording

In [ ]:
rows = []
for label, m in summary["tests"].items():
    fc = m.get("filter_counts", {})
    rows.append({
        "label": label,
        "raw": m["n_neurons_pre"],
        "kept": m["n_neurons"],
        "rej_circ": fc.get("circularity_rejected", 0),
        "rej_area": fc.get("max_area_rejected", 0),
        "rej_mask": fc.get("in_mask_rejected", 0),
        "recon_error": round(m["recon_error"], 6),
        "spatial_compact": round(m["spatial_compactness"], 4),
        "trace_sparsity": round(m["trace_sparsity"], 3),
        "composite": round(m["composite_score"], 5),
        "runtime_s": m["runtime_s"],
    })
pd.DataFrame(rows).set_index("label")

### 4.1 Test Half — Frames 26–52

CNMF initialised **32 raw components** on the held-out 27-frame segment; quality filter kept **2**.  
All 30 rejections were **max-area** (footprint > 201 px²). No circularity or out-of-mask rejections.  
Runtime: **~8 min**.

In [ ]:
display(Image(str(RUN_DIR / "contours_test_half_frames_26-52.png"), width=700))

The 2 kept neuron footprints are within the brain mask and passed the area filter. Their contours (cyan) are not clearly visible at this scale — footprints are near the 201 px² ceiling and small relative to the full 2048px FOV.

In [ ]:
display(Image(str(RUN_DIR / "traces_test_half_frames_26-52.png"), width=1000))

**N0** peaks ~70,000 AU around frame 10 then declines. **N1** starts near 200,000 AU and decays monotonically across the 27-frame window.  

Large absolute amplitudes and monotonic decay are consistent with **slow global signals** (neuropil drift or photobleaching) rather than single-neuron calcium transients. Zero temporal stability (`stability=0.0`) confirms that tune-half and test-half seeds do not overlap spatially — the model is not reproducing the same neurons across halves.

### 4.2 Full Movie — All 53 Frames

CNMF initialised **104 raw components**; quality filter kept **5**.  

| Rejection reason | Count | % of raw |
|---|---|---|
| Max-area | 71 | 68% |
| Circularity < 0.5 | 15 | 14% |
| Centroid outside mask | 13 | 13% |
| **Kept** | **5** | **5%** |

Runtime: **~37 min**.

In [ ]:
display(Image(str(RUN_DIR / "contours_full_movie.png"), width=700))

In [ ]:
display(Image(str(RUN_DIR / "traces_full_movie.png"), width=1050))

With 53 frames the traces show more structure:

- **N0, N1, N2** — large transient events peaking around frames 20–30 (up to ~65,000 AU). Timing is coordinated, consistent with a shared stimulus-driven rGECO response. N2 has a sharper onset around frame 25.
- **N3** — broad, slow dynamics peaking early (~frames 5–8) then gradually decaying; more consistent with a neuropil patch than a single soma.
- **N4** — smaller amplitude (~10,000 AU) with faster, irregular fluctuations; the closest candidate to single-neuron activity in this run.

Despite the more structured dynamics, high `trace_sparsity` (~4.8) and large absolute amplitudes still suggest ensemble-level signals rather than isolated cells.

---
## 5. Observations & Next Steps

### What worked
- **Brain mask** correctly isolates bilateral brain tissue; no bleed into the dark periphery
- **Stripe removal** cleanly suppresses the central column artifact (columns ~900–1300)
- **Bayesian search** found an interpretable regime (small `gSig`, large `rf`, `p=1`) by trial 4
- **Quality filters** cut ~93–95% of raw components; only plausible shapes survive

### Concerns

| Issue | Evidence | Likely cause |
|---|---|---|
| Very few neurons kept | Test: 2/32, Full: 5/104 | `max_area_factor=4.0` too tight at full res with `gSig=4` — ceiling ≈ 201 px² |
| Recon error ≈ 1.0 | Both test and full-movie | 2–5 components cannot reconstruct a dense 2048px brain; model under-fit |
| Zero stability | `stability=0.0` both tests | Tune/test seeds do not overlap — non-reproducible seeding, likely due to short T=26 |
| Large-amplitude traces | N0–N1 in test half (10⁴–10⁵ AU) | Surviving components are neuropil patches, not isolated somata |
| Early optimizer plateau | Flat from trial 4 onward | 10 calls vastly under-samples the 6D full-resolution search space |

### Suggested next steps

1. **Relax `--max-area-factor 10`** — at 2048px real soma occupy proportionally more pixels; the 201 px² ceiling rejects too many real neurons
2. **Increase `--n-calls 25 --n-initial 10`** — give the GP enough budget to explore the full-resolution space meaningfully
3. **Run `--resolution 512` comparison** — benchmark neuron yield and trace quality against the full-res result on the same data
4. **Verify `gSig` range** — `gSig=4` is an ~8px diameter at 2048px; check physical pixel size and expected soma diameter for rGECO-labelled zebrafish neurons (~10–15 µm)
5. **Confirm T=53** — only 53 timepoints makes seeding unreliable; verify this is the complete recording and not a truncated export